# 01 – H₂ Minimal Pipeline Demo

This notebook runs the simplest end-to-end pipeline:
```
PySCF HF  →  SimpleCAS  →  ParityMapper  →  NumPySolver (FCI)
```
and then swaps the solver to VQE-UCCSD to show solver exchangeability.

**Prerequisites**
```bash
pip install pyscf qiskit qiskit-nature qiskit-algorithms pyyaml
```
Make sure the `dft_qc_pipeline` package is importable from this notebook.

In [ ]:
import os, sys, pathlib

# Repo root = directory that contains `dft_qc_pipeline/` and `pyproject.toml`
_here = pathlib.Path().resolve()
for _root in [_here, _here.parent, _here.parent.parent]:
    if (_root / "pyproject.toml").is_file() and (_root / "dft_qc_pipeline").is_dir():
        sys.path.insert(0, str(_root))
        break

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

import numpy as np
import matplotlib.pyplot as plt

_NB = pathlib.Path().resolve()
_CFG_DIR = _NB.parent / "configs"

## 1. Load config and run pipeline (NumPy FCI solver)

The YAML file specifies the full pipeline; no code changes needed to swap methods.

In [ ]:
from dft_qc_pipeline import Pipeline, PipelineConfig

config = PipelineConfig.from_yaml(_CFG_DIR / "h2_vqe.yaml")
print('Backend:', config.backend.type, config.backend.method)
print('Embedding:', config.embedding.type)
print('Solver:', config.solver.type)

In [ ]:
pipeline = Pipeline(config)
result_fci = pipeline.run()

print(f'\nHF energy:      {result_fci.backend_result.energy_hf:.10f} Ha')
print(f'Fragment (FCI): {result_fci.total_energy:.10f} Ha')

## 2. Swap solver to VQE-UCCSD

Only the `solver.type` field changes; everything else is identical.

In [ ]:
config_vqe = PipelineConfig.from_yaml(_CFG_DIR / "h2_vqe.yaml")
config_vqe.solver.type = 'vqe'
config_vqe.solver.ansatz = 'uccsd'
config_vqe.solver.optimizer = 'slsqp'
config_vqe.solver.max_iter = 200

result_vqe = Pipeline(config_vqe).run()
print(f'VQE-UCCSD energy: {result_vqe.total_energy:.10f} Ha')
print(f'Error vs FCI:     {(result_vqe.total_energy - result_fci.total_energy)*1000:.4f} mHa')

## 3. H₂ Bond Dissociation Curve

Scan the H–H distance and compute FCI energy at each point using the pipeline.

In [ ]:
from dft_qc_pipeline import PipelineConfig, Pipeline

# In CI (`CI_FAST_NB=1`) scan a single geometry to keep nbmake fast.
if os.environ.get("CI_FAST_NB", "").strip():
    distances = np.array([0.735])
else:
    distances = np.linspace(0.5, 3.0, 20)
energies_hf = []
energies_fci = []

for d in distances:
    cfg = PipelineConfig.from_yaml(_CFG_DIR / "h2_vqe.yaml")
    cfg.backend.geometry = f'H 0 0 0; H 0 0 {d:.4f}'
    res = Pipeline(cfg).run()
    energies_hf.append(res.backend_result.energy_hf)
    energies_fci.append(res.total_energy)
    print(f'd={d:.2f} Å  HF={energies_hf[-1]:.6f}  FCI={energies_fci[-1]:.6f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(distances, energies_hf, 'b--o', ms=4, label='HF')
ax.plot(distances, energies_fci, 'r-o', ms=4, label='NumPy FCI (CAS)')
ax.set_xlabel('H–H distance (Å)')
ax.set_ylabel('Energy (Ha)')
ax.set_title('H₂ potential energy curve – pipeline demo')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('H2_PEC_pipeline.png', dpi=150)
plt.show()
if len(energies_fci) >= 5:
    print('Correlation energy at eq. (d≈0.74 Å):',
          f'{(energies_fci[4] - energies_hf[4])*1000:.2f} mHa')
else:
    print('Single-point demo (CI_FAST_NB): Δcorr not shown.')

## 4. Inspect the 1-RDM

The `NumPySolver` extracts the 1-RDM from the ground-state statevector.

In [ ]:
from dft_qc_pipeline.postprocessing import print_rdm1_summary

frag_res = result_fci.fragment_results.get('H2_full')
print_rdm1_summary(frag_res.rdm1 if frag_res else None, label='Fragment 1-RDM (FCI)')